# Comparable Company Analysis

**Purpose:** Demonstrate a compact peer-based relative-value workflow using the `finstack_quant.statements_analytics` comparable-company helpers.

**Prerequisites:** Familiarity with common credit and valuation ratios such as EV/EBITDA, leverage, interest coverage, and spread.

**In this notebook:** We place a subject issuer inside a peer set, compute ranks and z-scores, fit a spread-vs-leverage line, and combine several dimensions into one rich/cheap signal.


## Concept

A useful peer workflow usually answers three questions:

1. Where does the subject sit in the peer distribution?
2. What spread looks fair given one key driver, such as leverage?
3. Do multiple dimensions agree that the issuer looks rich or cheap?


In [ ]:
import json
from pathlib import Path

import sys
sys.path.insert(0, "../..")

from _shared import banner
from finstack_quant.statements_analytics import (
    compute_multiple,
    peer_stats,
    percentile_rank,
    regression_fair_value,
    score_relative_value,
    z_score,
)

_NOTEBOOK_DATA = json.loads(Path("data/comparable_company_analysis.json").read_text())

subject = {
    "ev_ebitda": 8.5,
    "leverage": 3.5,
    "interest_coverage": 4.2,
    "oas_bp": 350.0,
}

peers = _NOTEBOOK_DATA['peers']

metrics = ("ev_ebitda", "leverage", "interest_coverage", "oas_bp")
peer_series = {metric: [peer[metric] for peer in peers] for metric in metrics}


def company_metrics(cid, raw):
    """Build a canonical serde ``CompanyMetrics`` payload from a flat dict.

    ``leverage``/``interest_coverage``/``oas_bp`` are named fields on the
    Rust struct; ``ev_ebitda`` is a multiple, so it rides in ``custom``.
    """
    return {
        "id": cid,
        "attributes": {},
        "leverage": raw["leverage"],
        "interest_coverage": raw["interest_coverage"],
        "oas_bp": raw["oas_bp"],
        "custom": {"ev_ebitda": raw["ev_ebitda"]},
    }


peer_set = {
    "subject": company_metrics("SUBJECT", subject),
    "peers": [company_metrics(f"PEER-{i}", peer) for i, peer in enumerate(peers, start=1)],
    "period_basis": "ltm",
}


## Peer positioning and fair value

The next cell starts with a subject issuer, compares it against a ten-name peer set, and then blends the results into a composite relative-value score. This is a good notebook-scale example of how the analytics module can support a credit memo or screening process.


In [ ]:
banner("Subject company metrics")
for key, value in subject.items():
    print(f"{key:<20}: {value:>8.2f}")

subject_company = {"enterprise_value": 8_500.0, "ebitda": 1_000.0}
print(
    f"\ncompute_multiple(subject_company, 'ev_ebitda') = "
    f"{compute_multiple(subject_company, 'ev_ebitda'):.2f}x"
)

banner("Subject vs peers")
print(f"{'metric':<20} {'subject':>10} {'pctile':>10} {'z-score':>10}")
for metric in metrics:
    pctile = percentile_rank(peer_series[metric], subject[metric])
    z = z_score(peer_series[metric], subject[metric])
    print(f"{metric:<20} {subject[metric]:>10.2f} {pctile:>9.1f}% {z:>+10.2f}")

banner("Regression fair value")
reg = regression_fair_value(
    peer_series["leverage"],
    peer_series["oas_bp"],
    subject["leverage"],
    subject["oas_bp"],
)
print(f"slope              : {reg.slope:.2f} bps / turn")
print(f"intercept          : {reg.intercept:.2f} bps")
print(f"R-squared          : {reg.r_squared:.3f}")
print(f"fitted spread      : {reg.fitted_value:.1f} bps")
print(f"actual spread      : {subject['oas_bp']:.1f} bps")
print(f"residual           : {reg.residual:+.1f} bps")

banner("Peer distribution summary")
for metric in metrics:
    stats = peer_stats(peer_series[metric])
    print(
        f"{metric:<20} n={stats.count:>2} median={stats.median:.2f} "
        f"mean={stats.mean:.2f} std={stats.std_dev:.2f}"
    )

banner("Composite score")
dimensions = [
    {
        "label": "Spread vs Leverage",
        "y_extractor": {"named": "oas_bp"},
        "x_extractor": {"named": "leverage"},
        "weight": 0.50,
    },
    {
        "label": "Spread vs Coverage",
        "y_extractor": {"named": "oas_bp"},
        "x_extractor": {"named": "interest_coverage"},
        "weight": 0.30,
    },
    {
        "label": "EV/EBITDA",
        "y_extractor": {"custom": "ev_ebitda"},
        "x_extractor": None,
        "weight": 0.20,
        "direction": "higher_is_rich",
    },
]
score = score_relative_value(peer_set, dimensions)
print(f"company         : {score.company_id}")
print(f"composite_score : {score.composite_score:+.3f}")
print(f"confidence      : {score.confidence:.3f}")
print(f"peer_count      : {score.peer_count}")
for dim in score.dimensions:
    name = dim.label
    print(
        f"  {name:<20} pctile={dim.percentile * 100:>6.1f}% "
        f"z={dim.z_score:>+7.2f} weight={dim.weight:.2f}"
    )

## Takeaways

- `percentile_rank()` and `z_score()` answer slightly different questions and are both useful.
- `regression_fair_value()` is a compact way to anchor relative value on one driver.
- `score_relative_value()` takes a canonical `PeerSet` plus `ScoringDimension` specs, combining regression-anchored and distribution-based dimensions into one transparent composite signal.


In [ ]:
{
    "fair_spread_bp": round(reg.fitted_value, 1),
    "actual_spread_bp": round(subject['oas_bp'], 1),
    "spread_residual_bp": round(reg.residual, 1),
    "composite_score": round(score.composite_score, 3),
}
